# 03. Spectral 32-Color PBMC Cohort Walkthrough (`kreutmair_2026_icu_pbmc`)

In this tutorial, you will walk through a **32-color human ICU PBMC spectral cohort** (`kreutmair_2026_icu_pbmc`: 8 patients across 2 acquisition batches comparing `ICU_Trauma_VAP_ARDS_No` [$n=5$] vs `ICU_Trauma_VAP_ARDS_Yes` [$n=3$]).

**Why this dataset shines:**
- **Zero unassigned clusters:** All 22 FlowSOM metaclusters map cleanly into **10 canonical human PBMC populations** (`CD4+ Naive`, `CD4+ TCM`, `CD4+ TEMRA`, `CD8+ Naive / TEMRA`, `CD8+ Effector / Memory`, `CD3+ DN`, `NK Cells`, `Classical Monocytes`, `Naive B`, `Switched Memory B`).
- **Autofluorescence exclusion:** `markers.csv` tags `AF-A` as `autofluorescence` so it never leaks into transformation or clustering.
- **Active multi-batch correction:** `assess_batch_effect: true` detects severe inter-batch drift (`96.9 / 100`) and aligns distributions across batches.
- **Two-tier visual hierarchy:** `05_population_discovery/plots/clusters/` vs `plots/populations/` bridged by `cluster_to_population_bridge.png`.

> **Prerequisites (Download Raw `.fcs` Data):** Curated `metadata/` CSVs are included in the repository. Before running this notebook, download the raw `.fcs` files once from the terminal:
> ```bash
> flowx download kreutmair_2026_icu_pbmc
> ```

In [1]:
import os
from pathlib import Path

# Ensure working directory is the repository root whether launched from root or docs/tutorials/
if not Path("configs/config_run").exists() and Path("../../configs/config_run").exists():
    os.chdir("../..")
print("Working directory ready:", Path("configs/config_run").exists())

Working directory ready: True


## 1. Inspect Metadata & Config

```bash
flowx pipeline -c configs/config_run/spectral/kreutmair_2026_icu_pbmc.yaml --overwrite
```

In [2]:
import pandas as pd
import yaml
from pathlib import Path

samples_df = pd.read_csv("data/spectral/kreutmair_2026_icu_pbmc/metadata/samples.csv")
print("=== samples.csv ===")
print(samples_df[["sample_id", "condition", "batch_id", "total_events"]].to_string(index=False))

out_dir = Path("outputs/config_run/spectral/kreutmair_2026_icu_pbmc")
resolved = yaml.safe_load((out_dir / "resolved_config.yaml").read_text())
print("\nLineage markers resolved:", len(resolved["05_population_discovery"]["lineage_markers"]))

=== samples.csv ===
 sample_id               condition                batch_id  total_events
       186 ICU_Trauma_VAP_ARDS_Yes 2020-07-19_Cytek_Aurora        301633
       202  ICU_Trauma_VAP_ARDS_No 2020-07-19_Cytek_Aurora        282270
       235  ICU_Trauma_VAP_ARDS_No 2020-07-19_Cytek_Aurora        244640
       250 ICU_Trauma_VAP_ARDS_Yes 2020-07-19_Cytek_Aurora        436876
       253  ICU_Trauma_VAP_ARDS_No 2020-07-19_Cytek_Aurora        300531
       278  ICU_Trauma_VAP_ARDS_No 2020-07-19_Cytek_Aurora        232013
       311  ICU_Trauma_VAP_ARDS_No 2021-03-29_Cytek_Aurora        202930
       315 ICU_Trauma_VAP_ARDS_Yes 2021-03-29_Cytek_Aurora        238902

Lineage markers resolved: 31


## 2. Step 01 Time QC & Step 03 Cleanup Gating

FlowX executes `apply_time_qc()` followed by `arcsinh(cofactor=2000)` transformation and 4-stage cleanup gating (`Debris` $\to$ `Dual Singlets` $\to$ `FJComp-LIVE DEAD Blue-A` Viability):

In [3]:
cln = pd.read_csv(out_dir / "04_batch_preparation/tables/cohort_batch_summary.csv")
print(cln[["sample_id", "raw_event_count", "clean_event_count", "prepared_event_count", "retention_pct_of_raw"]].to_string(index=False))

 sample_id  raw_event_count  clean_event_count  prepared_event_count  retention_pct_of_raw
       186           301633             197173                 50000             16.576436
       202           282270             185879                 50000             17.713537
       235           244640              57935                 50000             20.438195
       250           436876             310827                 50000             11.444895
       253           300531             187647                 50000             16.637219
       278           232013             164684                 50000             21.550517
       311           202930             126849                 50000             24.639038
       315           238902             124837                 50000             20.929084


![Spectral PBMC Cleanup Diagnostics (Kreutmair 2026)](../images/tutorials/cleanup_spectral_pbmc_kreutmair.png)

## 3. Step 04B Batch Necessity Assessment & Normalization

Across the two ICU acquisition dates (`2020-07-19` and `2021-03-29`), `assess_batch_effect_need()` computes a Drift Score of `96.9 / 100.0` (`CAUTION_ANCHOR_FREE`) and executes quantile distribution alignment:

In [4]:
import json
ba = json.loads((out_dir / "04_batch_preparation/tables/batch_necessity_assessment.json").read_text())
print("Recommendation:", ba["recommendation"], "| Drift Score:", ba["drift_score"])

b_inv = pd.read_csv(out_dir / "04_batch_preparation/tables/batch_invariance_metrics.csv")
print(f"Mean EMD reduction: {b_inv['emd_reduction_pct'].mean():+.2f}% across {len(b_inv)} markers")

Recommendation: CAUTION_ANCHOR_FREE | Drift Score: 96.88
Mean EMD reduction: +99.33% across 32 markers


![Batch Necessity Summary (Kreutmair 2026)](../images/tutorials/batch_necessity_pbmc_kreutmair.png)

![Batch Density Comparison (Kreutmair 2026)](../images/tutorials/batch_density_pbmc_kreutmair.png)

![Batch EMD Reduction (Kreutmair 2026)](../images/tutorials/batch_emd_reduction_pbmc_kreutmair.png)

## 4. Step 05 Two-Tier Discovery: `clusters/` $\to$ Bridge $\to$ `populations/`

Notice how FlowX cleanly separates **technical metacluster diagnostics** (`plots/clusters/`) from **annotated biological populations** (`plots/populations/`):

In [5]:
pop_map = pd.read_csv(out_dir / "05_population_discovery/tables/cluster_to_population_map.csv")
print(f"Total clusters: {len(pop_map)} -> Unique biological PBMC populations: {pop_map['Annotated_Population'].nunique()}")
print(pop_map.groupby("Annotated_Population")[["Cluster_Label", "Event_Count"]].agg({"Cluster_Label": ", ".join, "Event_Count": "sum"}).to_string())

Total clusters: 22 -> Unique biological PBMC populations: 10
                                                               Cluster_Label  Event_Count
Annotated_Population                                                                     
CD3+ DN / Other T Cells                                                   C9         6380
CD4+ Central Memory T Cells (TCM)                           C3, C6, C12, C20        59289
CD4+ Naive T Cells                                                  C16, C19        28329
CD4+ TEMRA T Cells                                                        C0        18209
CD8+ Effector / Memory T Cells                                            C1        11257
CD8+ Naive / TEMRA T Cells                                           C7, C15        18317
Classical Monocytes (CD14+ CD16-)           C5, C10, C11, C13, C14, C18, C21       210086
NK Cells (CD56+)                                                      C2, C4        33186
Naive B Cells (CD19+ IgD+ CD27-)       

![Auto-k Consensus PAC Metrics (Kreutmair 2026)](../images/tutorials/discovery_auto_k_pbmc_kreutmair.png)

![UMAP Technical Clusters (Kreutmair 2026)](../images/tutorials/discovery_umap_clusters_pbmc_kreutmair.png)

![Cluster Expression Heatmap (Kreutmair 2026)](../images/tutorials/discovery_cluster_heatmap_pbmc_kreutmair.png)

![Cluster-to-Population Bridge (Kreutmair 2026)](../images/tutorials/discovery_bridge_pbmc_kreutmair.png)

![UMAP Biological PBMC Populations (Kreutmair 2026)](../images/tutorials/discovery_umap_pop_pbmc_kreutmair.png)

![Population Expression Heatmap (Kreutmair 2026)](../images/tutorials/discovery_pop_heatmap_pbmc_kreutmair.png)

![PBMC Marker Intensity Overlays (Kreutmair 2026)](../images/tutorials/discovery_umap_markers_pbmc_kreutmair.png)

![Condition-Stratified UMAP Density Contours (Kreutmair 2026)](../images/tutorials/discovery_umap_contours_pbmc_kreutmair.png)


## 5. Step 06 Differential State (DS) Testing

Comparing `ICU_Trauma_VAP_ARDS_Yes` vs `ICU_Trauma_VAP_ARDS_No` highlights functional marker shifts (`p = 0.0357`) across memory T-cell subsets:

In [6]:
ds_df = pd.read_csv(out_dir / "06_statistical_analysis/tables/differential_state_summary.csv")
print(ds_df.head(6)[["population", "marker", "mean_ref", "mean_target", "cohens_d", "p_value"]].to_string(index=False))

                       population                   marker  mean_ref  mean_target  cohens_d  p_value
               CD4+ TEMRA T Cells          FJComp-PE-Cy7-A    0.2859       0.3321    4.8533 0.035714
CD4+ Central Memory T Cells (TCM)          FJComp-BUV737-A    0.2697       0.2820    2.7507 0.071429
               CD4+ TEMRA T Cells FJComp-Alexa Fluor 700-A    0.3226       0.3589    2.6706 0.035714
       CD8+ Naive / TEMRA T Cells FJComp-Alexa Fluor 700-A    0.5174       0.5518    2.5505 0.035714
               CD4+ TEMRA T Cells           FJComp-BB660-A    0.4044       0.4358    2.5401 0.035714
 Naive B Cells (CD19+ IgD+ CD27-)          FJComp-BUV615-A    0.2810       0.2938    2.4165 0.035714


![Differential State Heatmap (Kreutmair 2026)](../images/tutorials/stats_ds_heatmap_pbmc_kreutmair.png)

## Next Steps

- Continue to **[04. Raw 64-Channel Spectral Unmixing Complement](04_raw_spectral_unmixing_complement.ipynb)** to see full-spectrum optical deconvolution from raw 64-detector Cytek Aurora APD channels.

## References & Scientific Standards

* **MIFlowCyt Standard:** [](doi:10.1002/cyto.a.20623)
* **FCS 3.1 Data File Standard:** [](doi:10.1002/cyto.a.20825)
* **Gating-ML 2.0 Standard:** [](doi:10.1002/cyto.a.22690)
* **FlowSOM Clustering:** [](doi:10.1002/cyto.a.22625)
* **CytoNorm Batch Normalization:** [](doi:10.1002/cyto.a.23904)
* **UMAP Dimensionality Reduction:** [](doi:10.48550/arXiv.1802.03426)
* **FlowKit Computational Toolkit:** [](doi:10.3389/fimmu.2021.768541)